# Mechanistic Interpretability of Vision-Language Models (VLMs)

This notebook is a companion to the **LLM SAE Tutorial**.  
We follow the same philosophy to build intuition through code, but ask a different scientific question:

> **In an LLM the central mystery is superposition** (many concepts crammed into one neuron).  
> **In a VLM the central mystery is modality fusion** (where and when does visual information become language?)

### Story arc

1. **VLM Architecture**: Three components, two modalities, one shared representation space.
2. **Logit Lens**: Apply the language model head to every intermediate layer and watch visual tokens "turn into words."
3. **Activation Patching**: Corrupt the image, restore one layer at a time, and find the causal bottleneck.
4. **Synthesis**: How does VLM interpretability compare to LLM interpretability? What is the VLM "dark matter"?

### References

- Logit Lens (original): [nostalgebraist, 2020](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)
- VLM interpretability survey: [ICLR Blogpost 2025](https://d2jud02ci9yv69.cloudfront.net/2025-04-28-vlm-understanding-29/blog/vlm-understanding/)
- Activation patching in VLMs: [Neo et al., 2024 *Towards a Mechanistic Understanding of LLaVA*](https://arxiv.org/abs/2401.15947)
- Logit lens applied to VLMs: [Huo et al., 2024 *MMNeuron*](https://arxiv.org/abs/2406.11193)
- LLaVA-1.5: [Liu et al., 2023](https://arxiv.org/abs/2310.03744) | PaliGemma: [Beyer et al., 2024](https://arxiv.org/abs/2407.07726)


## Connection to the LLM SAE Tutorial

| | LLM SAE Tutorial | This VLM Tutorial |
|---|---|---|
| **Central problem** | Superposition (polysemantic neurons) | Modality fusion (opaque visual-language integration) |
| **Primary tool** | Sparse Autoencoders (SAEs) | Logit Lens + Activation Patching |
| **Core experiment** | Feature steering via SAE directions | Layer-wise causal patching of visual information |
| **"Dark matter"** | Reconstruction error + KL divergence | Layers where visual info is not yet linguistic |
| **Hook point** | `blocks.L.hook_mlp_out` (TransformerLens) | `model.language_model.layers[L]` (HuggingFace) |

SAEs work beautifully for LLMs because there exist pretrained, downloadable SAE dictionaries.  
For VLMs, that ecosystem is still nascent.  
Activation patching and the logit lens require no pretrained dictionaries, only the model itself, making them ideal for studying VLMs today.


## 0.1 Imports and configurations

In [ ]:
import os; os.environ.setdefault('HF_HUB_DISABLE_XET', '1')  # avoid full-repo downloads
# ============================================================
# MODEL SELECTION — change this one line to switch the entire tutorial
# ============================================================

MODEL_FAMILY = "llava"  # "llava"  |  "paligemma"

# That's it. All downstream cells adapt automatically.

In [ ]:
# ============================================================
# Configuration dictionary — you are not required to understand
# every field right now. They are explained when first used.
# ============================================================

CONFIGS = {
    "llava": {
        # ── model & processor ──────────────────────────────────
        "model_name": "llava-hf/llava-1.5-7b-hf",
        "requires_login": False,
        # ── architecture labels (for printouts) ───────────────
        "vision_encoder_name": "CLIP ViT-L/14 @ 336px  (307 M params)",
        "connector_name": "2-layer MLP projector",
        "lm_name": "Vicuna-7B  (LLaMA-2 backbone, 32 transformer layers)",
        # ── vision token bookkeeping ──────────────────────────
        # CLIP ViT-L with 336px images and 14px patches → 24×24 = 576 patches.
        # Each patch becomes one visual token in the LM sequence.
        "n_image_tokens": 576,
        "image_size": 336,
        "patch_size": 14,
        # ── HuggingFace attribute paths ───────────────────────
        "lm_layers_path": "model.language_model.layers",
        "lm_norm_path": "model.language_model.norm",
        "lm_head_path": "lm_head",
        # ── prompt format ─────────────────────────────────────
        # LLaVA-1.5 uses a Vicuna-style chat template.
        # The processor.apply_chat_template() call builds this for us.
        "prompt_type": "chat",
        "question": "What animal is in this image? Answer in one word.",
        # ── target answer for patching experiment ─────────────
        # The model should generate "cat" (note leading space — LLaMA tokenizer).
        "target_word": "cat",  # try " cat"
    },
    "paligemma": {
        # ── model & processor ──────────────────────────────────
        "model_name": "google/paligemma-3b-mix-224",
        "requires_login": True,  # requires HuggingFace token
        # ── architecture labels ───────────────────────────────
        "vision_encoder_name": "SigLIP-So400M @ 224px  (400 M params)",
        "connector_name": "Linear projection",
        "lm_name": "Gemma-2B  (18 transformer layers)",
        # ── vision token bookkeeping ──────────────────────────
        # SigLIP with 224px images and 14px patches → 16×16 = 256 patches.
        "n_image_tokens": 256,
        "image_size": 224,
        "patch_size": 14,
        # ── HuggingFace attribute paths ───────────────────────
        "lm_layers_path": "model.language_model.layers",
        "lm_norm_path": "model.language_model.norm",
        "lm_head_path": "lm_head",
        # ── prompt format ─────────────────────────────────────
        # PaliGemma uses a prefix-style prompt (no system/user tags).
        "prompt_type": "prefix",
        "question": "answer en what animal is in this image?\n",
        # ── target answer ─────────────────────────────────────
        "target_word": "cat",
    },
}

cfg = CONFIGS[MODEL_FAMILY]
print(f"Config loaded  →  MODEL_FAMILY = '{MODEL_FAMILY}'")
print(f"Model: {cfg['model_name']}")
print(f"Vision encoder: {cfg['vision_encoder_name']}")
print(f"Connector: {cfg['connector_name']}")
print(f"Language model: {cfg['lm_name']}")
print(
    f"Image tokens: {cfg['n_image_tokens']}  "
    f"({cfg['image_size']}px / {cfg['patch_size']}px patch = "
    f"{cfg['image_size'] // cfg['patch_size']} × {cfg['image_size'] // cfg['patch_size']})"
)

Config loaded  →  MODEL_FAMILY = 'llava'
Model: llava-hf/llava-1.5-7b-hf
Vision encoder: CLIP ViT-L/14 @ 336px  (307 M params)
Connector: 2-layer MLP projector
Language model: Vicuna-7B  (LLaMA-2 backbone, 32 transformer layers)
Image tokens: 576  (336px / 14px patch = 24 × 24)


In [ ]:
# HuggingFace login — required only for PaliGemma (gated model).
# LLaVA-1.5-7b is publicly accessible; no token needed.
if cfg["requires_login"]:
    from huggingface_hub import login

    login()  # paste your HuggingFace token when prompted
else:
    print(f"{cfg['model_name']} is publicly available — no login required.")

llava-hf/llava-1.5-7b-hf is publicly available — no login required.


In [ ]:
import contextlib
import random
from functools import reduce

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, LlavaForConditionalGeneration, PaliGemmaForConditionalGeneration


# ── reproducibility ──────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ── device ───────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

---
# 1) VLM Architecture: Three Components, Two Modalities

Before we can interpret a model, we need to understand its structure.

A modern VLM is assembled from three largely independent components:

```
┌─────────────────────────────────────────────────────┐
│                                                     │
│   Image ──► [ Vision Encoder ] ──► image features   │
│                                           │         │
│                                    [ Connector ]    │
│                                           │         │
│   Text  ──► [ Tokenizer + Embed ] ──► text embeds   │
│                                           │         │
│             (image + text embeddings concatenated)  │
│                                           │         │
│                             [ Language Model (LM) ] │
│                                           │         │
│                                          output     │
└─────────────────────────────────────────────────────┘
```

### Component 1: Vision Encoder
A Vision Transformer (ViT) that takes the raw image and produces a sequence of patch embeddings.  
Each patch (e.g., 14×14 pixels) becomes one vector.

The number of visual tokens depends on the image resolution and patch size:
$$n\_\text{patches} = \left(\frac{H}{\text{patch_size}}\right) \times \left(\frac{W}{\text{patch_size}}\right)$$

| Model | Vision Encoder | Image Size | Patch Size | Visual Tokens |
|---|---|---|---|---|
| **LLaVA-1.5-7b** | CLIP ViT-L14 | 336 × 336 px | 14 px | 24 × 24 = **576** |
| **PaliGemma-3b** | SigLIP-So400M | 224 × 224 px | 14 px | 16 × 16 = **256** |

The vision encoder lives in a completely separate parameter space from the language model.  
It is typically a pretrained CLIP or SigLIP model that has learned to align images with text captions.

### Component 2: Connector (Projector)
A small network (MLP or linear layer) that **bridges the dimensionality gap** between the vision encoder output and the LM's embedding space.  

The exact dimensions depend on the model:

| Model | Vision encoder output dim | LM embedding dim (d_model) | Connector type |
|---|---|---|---|
| **LLaVA-1.5-7b** | 1024 | 4096 | 2-layer MLP |
| **PaliGemma-3b** | 1152 | 2048 | Linear projection |

The connector is the only part of the architecture trained from scratch to connect vision and language.

### Component 3: Language Model
A standard autoregressive transformer (LLaMA, Gemma, etc.) that processes a *mixed sequence* of:
- **Visual tokens**: the connector's output, one per image patch  
- **Text tokens**: the usual token embeddings from the tokenizer  

The LM has no inherent knowledge that some tokens came from an image. It treats them as any other sequence.

### The interpretability challenge

This three-component structure creates a question that has **no analogue in pure LLMs**:

> *At which layer of the language model does the visual information get "understood"?  
> Does the LM start making language-like predictions about visual tokens immediately, or only in later layers?*

This is exactly what Logit Lens and Activation Patching will help us answer.

**References**:
- LLaVA-1.5 architecture: [Liu et al., 2023](https://arxiv.org/abs/2310.03744)
- PaliGemma architecture: [Beyer et al., 2024](https://arxiv.org/abs/2407.07726)
- General ViT background: [Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929)


## 1.1 Load model and processor

In [ ]:
# ============================================================
# 1.1  Load model + processor
# ============================================================
# `device_map="auto"` lets HuggingFace shard across available GPUs/CPU.
# `torch_dtype=torch.bfloat16` halves memory usage with negligible quality loss.
# On a single L4 GPU:
#   LLaVA-1.5-7b  ≈ 14 GB  (fits with ~10 GB headroom for activations)
#   PaliGemma-3b  ≈  6 GB  (very comfortable)

print(f"Loading {cfg['model_name']} …  (this may take 1-3 minutes on first run)")

if MODEL_FAMILY == "llava":
    model = LlavaForConditionalGeneration.from_pretrained(
        cfg["model_name"],
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
else:  # paligemma
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        cfg["model_name"],
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

processor = AutoProcessor.from_pretrained(cfg["model_name"])
model.eval()  # disable dropout; we only do inference

print("\nModel loaded successfully.")
print(f"Total parameters : {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B")

Loading llava-hf/llava-1.5-7b-hf …  (this may take 1-3 minutes on first run)



Model loaded successfully.
Total parameters : 7.06 B


In [ ]:
# Debug 1
def get_nested_attr(obj, path: str):
    return reduce(getattr, path.split("."), obj)


candidates = [
    "model.language_model",
    "model.language_model.model",
    "model.language_model.model.layers",
    "model.language_model.model.norm",
    "model.language_model.layers",
    "model.language_model.norm",
    "model.vision_tower",
    "model.multi_modal_projector",
]

for path in candidates:
    try:
        obj = get_nested_attr(model, path)
        print(f"[OK] {path}: {type(obj)}")
    except Exception as e:
        print(f"[FAIL] {path}: {e}")

[OK] model.language_model: <class 'transformers.models.llama.modeling_llama.LlamaModel'>
[FAIL] model.language_model.model: 'LlamaModel' object has no attribute 'model'
[FAIL] model.language_model.model.layers: 'LlamaModel' object has no attribute 'model'
[FAIL] model.language_model.model.norm: 'LlamaModel' object has no attribute 'model'
[OK] model.language_model.layers: <class 'torch.nn.modules.container.ModuleList'>
[OK] model.language_model.norm: <class 'transformers.models.llama.modeling_llama.LlamaRMSNorm'>
[OK] model.vision_tower: <class 'transformers.models.clip.modeling_clip.CLIPVisionModel'>
[OK] model.multi_modal_projector: <class 'transformers.models.llava.modeling_llava.LlavaMultiModalProjector'>


In [ ]:
# Debug 2
candidates = [
    "model",
    "model.layers",
    "model.norm",
    "lm_head",
    "text_model",
    "text_model.layers",
    "language_model",
    "language_model.model",
]

for path in candidates:
    try:
        obj = get_nested_attr(model, path)
        print(f"[OK] {path}: {type(obj)}")
    except Exception as e:
        print(f"[FAIL] {path}: {e}")

[OK] model: <class 'transformers.models.llava.modeling_llava.LlavaModel'>
[FAIL] model.layers: 'LlavaModel' object has no attribute 'layers'
[FAIL] model.norm: 'LlavaModel' object has no attribute 'norm'
[OK] lm_head: <class 'torch.nn.modules.linear.Linear'>
[FAIL] text_model: 'LlavaForConditionalGeneration' object has no attribute 'text_model'
[FAIL] text_model.layers: 'LlavaForConditionalGeneration' object has no attribute 'text_model'
[FAIL] language_model: 'LlavaForConditionalGeneration' object has no attribute 'language_model'
[FAIL] language_model.model: 'LlavaForConditionalGeneration' object has no attribute 'language_model'


## 1.2 Token coexistence: how visual and text tokens share the sequence

At a high level, most VLMs do the same thing:

1. Encode the image into a sequence of patch-level visual representations.
2. Map those representations into the language model hidden size.
3. Insert or align them with the token sequence processed by the LM.

The exact bookkeeping differs across architectures.
For example, LLaVA and PaliGemma do not handle image tokens in exactly the same way.

The LM then receives a flat sequence. The exact layout depends on the model:

**LLaVA-1.5** (chat-style, image token in the middle of the prompt):
```
[BOS] [SYS tokens ...] [visual_0] [visual_1] ... [visual_575] [text_0] [text_1] ... [EOS]
                        ↑────────────────────────────────────↑
                                  576 visual tokens (LLaVA)                           
```

**PaliGemma** (prefix-style, all image tokens come first):
```
[visual_0] [visual_1] ... [visual_255] [text_0] [text_1] ... [EOS]
↑──────────── 256 visual tokens ──────↑
```

In this notebook we are running **`MODEL_FAMILY = "{MODEL_FAMILY}"`**, which uses the
{'first' if cfg['prompt_type'] == 'prefix' else 'mixed'} layout above.

The LM layers have **no explicit flag** distinguishing visual from text positions.  
They process all tokens with the same attention mechanism.

> **Key insight for interpretability**: if we apply the LM's unembedding matrix (logit lens) to the hidden state at a *visual* token position, we are asking: *"what word would the model predict if this visual token were the final token?"*  
> Tracking this across layers reveals how the model progressively makes sense of visual information.


In [ ]:
# ============================================================
# 1.2  Inspect the three-component structure
# ============================================================
# We use a helper to access nested attributes by dotted path.


def get_nested_attr(obj, path: str):
    """Access a nested attribute via a dotted string, e.g. 'a.b.c'."""
    return reduce(getattr, path.split("."), obj)


lm_layers = get_nested_attr(model, cfg["lm_layers_path"])
lm_norm = get_nested_attr(model, cfg["lm_norm_path"])
lm_head = get_nested_attr(model, cfg["lm_head_path"])

n_layers = len(lm_layers)
d_model = lm_layers[0].self_attn.q_proj.in_features
vocab_size = lm_head.weight.shape[0]

print("=" * 55)
print("Component 1 — Vision Encoder")
print(f"{cfg['vision_encoder_name']}")
print()
print("Component 2 — Connector")
print(f"{cfg['connector_name']}")
print(f"image patches → {cfg['n_image_tokens']} visual tokens")
print()
print("Component 3 — Language Model")
print(f"{cfg['lm_name']}")
print(f"Transformer layers: {n_layers}")
print(f"d_model: {d_model}")
print(f"Vocabulary size: {vocab_size:,}")
print("=" * 55)

Component 1 — Vision Encoder
CLIP ViT-L/14 @ 336px  (307 M params)

Component 2 — Connector
2-layer MLP projector
image patches → 576 visual tokens

Component 3 — Language Model
Vicuna-7B  (LLaMA-2 backbone, 32 transformer layers)
Transformer layers: 32
d_model: 4096
Vocabulary size: 32,064


## 1.3 Read or upload input image
Image from Wikipedia article Tabby cat (https://en.wikipedia.org/wiki/Tabby_cat)

In [ ]:
# ============================================================
# 1.3  Upload your own image to Google Colab and inspect the token sequence
# For the saved example output, we use a cat image because it gives a clean target token for patching.
# ============================================================

# from io import BytesIO
# uploaded = files.upload()

# # Take the first uploaded file
# filename = next(iter(uploaded))
# image = Image.open(BytesIO(uploaded[filename])).convert("RGB")

# print(f"Loaded file: {filename}")
# print(f"Image size: {image.size[0]} × {image.size[1]} px")

# fig, ax = plt.subplots(figsize=(4, 4))
# ax.imshow(image)
# ax.axis("off")
# ax.set_title(f"Uploaded image: {filename}", fontsize=11)
# plt.tight_layout()
# plt.show()

Saving cat.jpg to cat (1).jpg
Loaded file: cat (1).jpg
Image size: 1795 × 2397 px


In [ ]:
# ============================================================
# Read image from local
# ============================================================

# Local image path
image_path = "./data/cat.jpg"

# Open image
image = Image.open(image_path).convert("RGB")

print(f"Loaded file: {image_path}")
print(f"Image size: {image.size[0]} × {image.size[1]} px")

# Display image
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image)
ax.axis("off")
ax.set_title("Loaded image: cat.jpg", fontsize=11)

plt.tight_layout()
plt.show()

## 1.4 Build input

In [ ]:
# ============================================================
# 1.4  Build the input and visualise the token-type layout
# ============================================================

QUESTION = cfg["question"]

# ── Build prompt string ──────────────────────────────────────
if cfg["prompt_type"] == "chat":
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": QUESTION},
            ],
        },
    ]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
else:
    prompt_text = QUESTION

# ── Tokenize ─────────────────────────────────────────────────
inputs = processor(
    text=prompt_text,
    images=image,
    return_tensors="pt",
).to(device)

input_ids = inputs["input_ids"]
print(f"input_ids length: {input_ids.shape[1]}")

image_token_id = model.config.image_token_index
img_positions = (input_ids[0] == image_token_id).nonzero(as_tuple=False).flatten()

print(f"Number of <image> tokens in input_ids: {len(img_positions)}")

# ── Run one forward pass to get the REAL sequence length ─────
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

hidden_states = outputs.hidden_states
seq_len_actual = hidden_states[0].shape[1]
print(f"Actual hidden-state sequence length: {seq_len_actual}")

# ── Build visual mask safely ─────────────────────────────────
visual_mask = torch.zeros(seq_len_actual, dtype=torch.bool)

if len(img_positions) == 0:
    raise ValueError("No image token found in input_ids.")

elif len(img_positions) == seq_len_actual:
    # Rare / unlikely, but included for completeness
    visual_mask[:] = True

elif len(img_positions) > 1:
    # Image tokens are already expanded in input_ids
    img_start = img_positions[0].item()
    img_end = img_positions[-1].item() + 1

    if img_end > seq_len_actual:
        raise ValueError(f"Image-token span [{img_start}, {img_end}) exceeds actual sequence length {seq_len_actual}.")

    visual_mask[img_start:img_end] = True
    print(f"Using expanded image-token span in input_ids: [{img_start}, {img_end})")

else:
    # Single placeholder case.
    # We cannot blindly trust cfg["n_image_tokens"].
    # Infer visual-token count from actual sequence length if possible.
    img_start = img_positions[0].item()

    inferred_n_img = seq_len_actual - (input_ids.shape[1] - 1)
    if inferred_n_img <= 0:
        raise ValueError(f"Inferred number of visual tokens is invalid: {inferred_n_img}")

    img_end = img_start + inferred_n_img
    if img_end > seq_len_actual:
        raise ValueError(
            f"Inferred image-token span [{img_start}, {img_end}) exceeds actual sequence length {seq_len_actual}."
        )

    visual_mask[img_start:img_end] = True
    print(f"Single placeholder at position {img_start}")
    print(f"Inferred visual-token span: [{img_start}, {img_end})")
    print(f"Inferred number of visual tokens: {inferred_n_img}")

text_mask = ~visual_mask

print(f"Visual tokens: {visual_mask.sum().item()}")
print(f"Text tokens: {text_mask.sum().item()}")
print(f"Mask length: {len(visual_mask)}")

input_ids length: 600
Number of <image> tokens in input_ids: 576
Actual hidden-state sequence length: 600
Using expanded image-token span in input_ids: [5, 581)
Visual tokens: 576
Text tokens: 24
Mask length: 600


---
# 2) Logit Lens: Watching Visual Tokens Become Words

## 2.1 The core idea

In the LLM SAE Tutorial we asked: what feature does a neuron represent?  
Here we ask a different question: what word does a token position represent at layer L?

The **logit lens** answers this by applying the model's final unembedding step to intermediate hidden states.

Formally, for layer $L$ at token position $t$:

$$\text{logit-lens}_L(t) = W_{\text{unembed}} \cdot \text{LayerNorm}(h_L^{(t)})$$


> This is a useful approximation for tutorial purposes.
The exact readout path can differ slightly across architectures, especially in multimodal models.

where $h_L^{(t)}$ is the hidden state after transformer layer $L$, and $W_{\text{unembed}}$ is the model's final linear projection to vocabulary logits.

This gives us a full vocabulary distribution at every layer for every token, including visual tokens.

### Why this is interesting for VLMs

In a pure LLM, every token is already a word, so the logit lens tracks how the model refines its prediction.  
In a VLM, visual tokens start as image patches with no linguistic meaning. Applying the logit lens to them reveals how linguistic structure emerges layer by layer.


A commonly observed pattern (documented for LLaVA-style models) has three stages:

1. **Early layers** high entropy: the model cannot yet "read" the visual token in linguistic terms.
2. **Middle layers** entropy drops sharply: the model starts mapping visual tokens to specific word concepts.
3. **Late layers** slight entropy increase: representations become contextual, less locally constrained.

> ⚠️ **Model-specific note**: this three-stage pattern was documented empirically for LLaVA-NeXT (32-layer backbone) in [Huo et al., 2024](https://arxiv.org/abs/2406.11193) and LLaVA-1.5 in [Neo et al., 2024](https://arxiv.org/abs/2401.15947). For PaliGemma (18-layer Gemma-2B backbone), the entropy curve may look different. The minimum can appear as early as layer 3. Observe what your plot actually shows before drawing conclusions.


> **Connection to the LLM notebook**: in the SAE tutorial we measured *reconstruction quality* (variance explained) as a proxy for how much of the model's computation our tool captures. Here, entropy is the analogous diagnostic. It tells us how much "linguistic signal" the model has extracted from visual tokens at each depth.

**Entropy definition**:  
$H_L(t) = -\sum_{v} p_v \log p_v$ where $p = \text{softmax}(\text{logit-lens}_L(t))$

Low entropy = model is confident about a specific word. High entropy = distribution is spread across many words.


In [ ]:
# ============================================================
# 2.2  Define the logit-lens helper
# ============================================================
@torch.no_grad()
def logit_lens(hidden_state, layer_norm, lm_head):
    target_dtype = lm_head.weight.dtype
    target_device = lm_head.weight.device

    x = hidden_state.to(device=target_device, dtype=target_dtype)
    x = layer_norm(x)
    logits = lm_head(x)
    return torch.softmax(logits.float(), dim=-1)


def entropy(probs: torch.Tensor) -> torch.Tensor:
    """
    Shannon entropy H = -sum(p * log p).

    Args:
        probs : [..., vocab_size]

    Returns
    -------
        H     : [...]  in nats
    """
    # clamp to avoid log(0)
    return -(probs * torch.clamp(probs, min=1e-10).log()).sum(dim=-1)


print("Helper functions defined: logit_lens(), entropy()")

In [ ]:
# ============================================================
# 2.3  Forward pass with output_hidden_states=True
# ============================================================
# `output_hidden_states=True` returns a tuple of hidden state tensors,
# one per layer (plus the embedding layer), at every token position.
#
# hidden_states[0] → embedding output  (before any transformer layer)
# hidden_states[1] → output of transformer layer 0
# hidden_states[L+1 → output of transformer layer L
# hidden_states[N] → output of the final transformer layer

print("Running forward pass (this may take 20–40 s on first run) …")

with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
    )

hidden_states = outputs.hidden_states  # tuple of (n_layers+1) tensors
n_hs = len(hidden_states)  # n_layers + 1

print(f"Number of hidden state snapshots : {n_hs}  (= 1 embedding + {n_hs - 1} transformer layers)")
print(f"Shape of each snapshot: {hidden_states[0].shape}")  # [1, seq_len_merged, d_model]

Running forward pass (this may take 20–40 s on first run) …
Number of hidden state snapshots : 33  (= 1 embedding + 32 transformer layers)
Shape of each snapshot: torch.Size([1, 600, 4096])


In [ ]:
print("input_ids length:", input_ids.shape[1])
print("hidden_states[0] shape:", hidden_states[0].shape)
print("number of image-token placeholders in input_ids:", (input_ids[0] == model.config.image_token_index).sum().item())
print("cfg['n_image_tokens']:", cfg["n_image_tokens"])

# sanity check
expected_seq_len = hidden_states[1].shape[1]
assert len(visual_mask) == expected_seq_len, (
    f"visual_mask length {len(visual_mask)} != hidden-state seq len {expected_seq_len}"
)

input_ids length: 600
hidden_states[0] shape: torch.Size([1, 600, 4096])
number of image-token placeholders in input_ids: 576
cfg['n_image_tokens']: 576


In [ ]:
# ============================================================
# 2.4  Compute per-layer entropy for visual vs text tokens
# ============================================================

lm_norm = get_nested_attr(model, cfg["lm_norm_path"])
lm_head = get_nested_attr(model, cfg["lm_head_path"])

# We skip hidden_states[0] (embedding layer; no transformer has run yet)
# and iterate over layers 1 … n_layers.

entropy_visual = []  # mean entropy across visual token positions, per layer
entropy_text = []  # mean entropy across text token positions, per layer

# Move masks to CPU for indexing (hidden states may be on GPU)
vis_mask_cpu = visual_mask.cpu()  # [seq_len_merged]
text_mask_cpu = text_mask.cpu()

print("Computing logit-lens entropy for each layer …")
for hs in tqdm(hidden_states[1:], desc="Layers"):  # skip embedding layer
    # hs shape: [1, seq_len_merged, d_model]
    probs = logit_lens(hs, lm_norm, lm_head)  # [1, seq_len_merged, vocab]
    H = entropy(probs[0].cpu())  # [seq_len_merged]

    entropy_visual.append(H[vis_mask_cpu].mean().item())
    entropy_text.append(H[text_mask_cpu].mean().item())

layer_indices = list(range(1, len(entropy_visual) + 1))  # 1-indexed
print("Done.")

Computing logit-lens entropy for each layer …


Done.


In [ ]:
# ============================================================
# 2.5  Plot: entropy across layers for visual vs text tokens
# ============================================================

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    layer_indices,
    entropy_visual,
    color="steelblue",
    linewidth=2.5,
    marker="o",
    markersize=4,
    label="Visual tokens (image patches)",
)
ax.plot(layer_indices, entropy_text, color="coral", linewidth=2.5, marker="s", markersize=4, label="Text tokens")

ax.set_xlabel("Transformer layer index", fontsize=11)
ax.set_ylabel("Mean entropy (nats)", fontsize=11)
ax.set_title(
    f"Logit-lens entropy across layers — {cfg['model_name'].split('/')[-1]}\n"
    "Visual-token logit-lens entropy across layers",
    fontsize=11,
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, len(entropy_visual) + 1, max(1, len(entropy_visual) // 10)))
plt.tight_layout()
plt.show()

# Print the layer at which visual entropy is minimum
min_layer = layer_indices[int(np.argmin(entropy_visual))]
min_entropy = min(entropy_visual)
print(f"\nVisual entropy minimum at layer {min_layer}  (= {min_entropy:.5f} nats)")
print(f"Text entropy at same layer        : {entropy_text[min_layer - 1]:.5f} nats")


Visual entropy minimum at layer 31  (= 3.34997 nats)
Text entropy at same layer        : 2.21059 nats


> These two curves answer different questions.
Entropy is a correlational readability signal.
Restoration is a causal intervention signal.
Agreement is interesting, but mismatch is also informative.

In [ ]:
# ── Automatically divide into three stages and print. ─────────────────────────────────────
min_idx = int(np.argmin(entropy_visual))  # 0-indexed in entropy list
# Heuristic: "crystallization" = layers where entropy is within 20% of min
low_thresh = min_entropy * 1.20
crystal_layers = [layer_indices[i] for i, e in enumerate(entropy_visual) if e <= low_thresh]

print("\nAutomatic stage estimation (heuristic, ±1 layer):")
print(f"Alignment stage: layers 1 – {crystal_layers[0] - 1 if crystal_layers[0] > 1 else 1}")
print(f"Crystallization: layers {crystal_layers[0]} – {crystal_layers[-1]}")
if crystal_layers[-1] < layer_indices[-1]:
    print(f"Broadening stage: layers {crystal_layers[-1] + 1} – {layer_indices[-1]}")
else:
    print("Broadening stage: not clearly present in this model/image")


Automatic stage estimation (heuristic, ±1 layer):
Alignment stage: layers 1 – 30
Crystallization: layers 31 – 31
Broadening stage: layers 32 – 32


## 2.6 What word does a visual token "look like" at each layer?

Beyond entropy, we can ask **which specific word** the logit-lens assigns highest probability to at each layer for a single visual token.

For a visual token corresponding to a patch in the image (e.g., the cat's face region), we expect:
- **Early layers**: generic, low-frequency tokens, the model has no idea.
- **Middle layers**: concrete nouns related to the visual content (cat, animal, fur, …).
- **Late layers**: increasingly context-dependent predictions.

**What the literature reports for LLaVA**: [Neo et al., 2024](https://arxiv.org/abs/2401.15947) found that in LLaVA-1.5, late-layer visual token representations tend to correspond to vocabulary tokens that semantically describe the patch content (e.g., a patch covering a cat's face might decode to "cat", "fur", "feline" in later layers).


**What you may observe with PaliGemma**: The pattern can look quite different. Because PaliGemma uses:
- A **SigLIP** vision encoder (trained with a different loss than CLIP), and
- A more compact **18-layer** Gemma-2B backbone, the top-k decoded tokens at visual positions may include non-English fragments, rare tokens, or tokens that appear unrelated to the image semantics. This does **not** mean the model is broken. It means that the linguistic-semantic information is distributed differently across the vocabulary manifold for this architecture.

> **The key signal is entropy, not the specific tokens**: even if top-1 decoded tokens are not recognizable words, the *entropy drop* across layers is still informative. It tells us that the model is becoming more confident about some direction in vocabulary space, even if that direction doesn't align with the obvious human-readable word.

Run the cell below and observe what you find for your specific image and model.


In [ ]:
# ============================================================
# 2.7  Top decoded tokens per layer for a single visual token
# ============================================================
# We pick the visual token in the "middle" of the visual region
# as a representative patch (likely somewhere on the main subject).
# For simplicity we probe a middle visual token.
# A stronger analysis would map patch indices back to image regions and choose a token that clearly overlaps the object of interest.

vis_positions = vis_mask_cpu.nonzero(as_tuple=False).squeeze(-1).tolist()
probe_pos = vis_positions[len(vis_positions) // 2]  # central visual token

print(f"Probing token position {probe_pos} (middle of the visual region)\n")

tokenizer = processor.tokenizer
top_k = 5

# Display top-5 tokens at each layer
STEP = max(1, len(hidden_states[1:]) // 8)  # show ~8 layer snapshots

print(f"{'Layer':>6} | {'Entropy':>8} | Top-{top_k} predicted words")
print("-" * 70)

for i, hs in enumerate(hidden_states[1:]):  # skip embedding layer
    if i % STEP != 0 and i != len(hidden_states) - 2:
        continue
    probs = logit_lens(hs, lm_norm, lm_head)  # [1, seq, vocab]
    p_pos = probs[0, probe_pos].cpu()  # [vocab]
    H_val = entropy(p_pos.unsqueeze(0)).item()

    top_ids = torch.topk(p_pos, top_k).indices.tolist()
    top_tks = [repr(tokenizer.decode([tid])) for tid in top_ids]
    top_ps = [f"{p_pos[tid].item():.3f}" for tid in top_ids]

    words_str = "  ".join(f"{t}({p})" for t, p in zip(top_tks, top_ps))
    print(f" L{i + 1:>3}  | {H_val:>8.3f} | {words_str}")

Probing token position 293 (middle of the visual region)

 Layer |  Entropy | Top-5 predicted words
----------------------------------------------------------------------
 L  1  |    8.551 | 'amos'(0.008)  'aks'(0.008)  'ommes'(0.007)  'correspond'(0.007)  'conf'(0.005)
 L  5  |    8.464 | 'correspond'(0.012)  'amos'(0.011)  'pios'(0.007)  'Meister'(0.007)  'inos'(0.006)
 L  9  |    8.415 | 'iele'(0.017)  'gresql'(0.013)  'empio'(0.012)  'Æ'(0.009)  'opp'(0.008)
 L 13  |    8.609 | '식'(0.015)  'quier'(0.006)  '土'(0.006)  'ittel'(0.006)  'ouvel'(0.005)
 L 17  |    8.258 | 'close'(0.023)  'unos'(0.013)  'Abb'(0.013)  'magn'(0.012)  'background'(0.011)
 L 21  |    7.572 | 'close'(0.094)  'another'(0.037)  'magn'(0.032)  'view'(0.031)  'gresql'(0.018)
 L 25  |    6.426 | 'a'(0.300)  'another'(0.027)  'an'(0.024)  'sky'(0.011)  'clouds'(0.010)
 L 29  |    0.756 | 'a'(0.916)  'an'(0.014)  'the'(0.012)  'its'(0.009)  'another'(0.003)
 L 32  |    5.192 | 'the'(0.101)  'a'(0.095)  'cat'(0.033) 

## 2.8 Interpreting the three-stage pattern

The entropy curve reveals how linguistic structure builds up in the LM across layers.
Researchers have described a **three-stage lifecycle** for visual token representations,
though the exact layer ranges depend heavily on model architecture and backbone depth:


| Stage | Layers | What is happening | Entropy |
|---|---|---|---|
| **Feature alignment** | Early layers | Visual tokens carry patch-level pixel statistics; the LM head maps these to unpredictable or rare vocabulary tokens | Very high; model is "guessing" |
| **Concept crystallization** | Middle layers | The LM progressively compresses visual information into language-aligned directions | Sharply falling |
| **Contextual broadening** | Late layers | Representations become contextual; the model spreads probability over semantically related words | Slight increase or plateau |

> **Model-specific calibration**: this three-stage pattern was originally described for large
> VLMs with 32-layer LLM backbones (LLaVA-1.5, LLaVA-NeXT). For a model with less layers, the middle stage may begin much earlier. Look at your entropy plot and identify where the drop actually starts and ends. Let the data guide the stage boundaries, not the table above.


This pattern has two important implications:

1. **The LM backbone does real work on visual tokens**: It is not the case that only the connector matters. Significant linguistic structure emerges across many transformer layers.

2. **There is a "linguistic sweet spot"**: The layer of minimum visual entropy is where visual tokens are most language-like. This is a natural candidate for probing or intervention for VLMs.

> **Contrast with the LLM notebook**: In the LLM tutorial, *text* tokens had low entropy throughout (the model always has a reasonable word prediction). Here, *visual* tokens start with very high entropy. They are genuinely opaque to the language head at first. The drop in entropy is the model's process of making sense of the image.


---
# 3) Activation Patching: Finding the Causal Bottleneck

## 3.1 Motivation: correlation is not causation

The logit lens showed us **correlational** evidence. The Visual tokens *look like* specific words at certain layers.  
But does that mean those layers are *causally responsible* for the model's correct answer?

**Activation patching** answers this with a controlled experiment:

1. Run the model on a **clean input** (original image + question) → correct answer.
2. Run the model on a **corrupted input** (Gaussian noise replacing the image) → wrong or uncertain answer.
3. For each layer $L$: *run the corrupted forward pass, but replace the hidden states at layer $L$ with those from the clean run.*
4. Measure how much the correct-answer probability is **restored** by the patch at layer $L$.

If patching layer $L$ fully restores the clean answer, then layer $L$ is a **sufficient location** for the visual information to be present — it is causally upstream of the correct behavior.

The result is a **restoration curve**: $\text{restoration}(L) \in [0, 1]$ across layers.

A peak in the restoration curve tells us: *"layer $L$ is where the model's processing of visual information critically determines the output."*

### References

- Original activation patching for LLMs: [Meng et al., 2022 (ROME)](https://arxiv.org/abs/2202.05262)
- Applied to VLMs (LLaVA): [Neo et al., 2024](https://arxiv.org/abs/2401.15947), specifically their "causal tracing" experiments
- Also related to [Palit et al., 2023 (BLIP analysis)](https://arxiv.org/abs/2312.07893)

> **Connection to the LLM notebook**: in the SAE tutorial we used *feature steering* as a causal probe — we added a direction and watched the output change. Activation patching is the *complementary* causal probe: we *remove* visual information (via corruption) and *restore* it one layer at a time.


In [ ]:
# ============================================================
# 3.2  Create corrupted input (Gaussian noise image)
# ============================================================
# We replace the image with random Gaussian noise (same size, same dtype).
# The text prompt remains identical.
# The model should now fail to answer correctly because
# no real visual information is present.


def make_corrupted_inputs(inputs_clean: dict) -> dict:
    """Return a copy of inputs where pixel_values is replaced by Gaussian noise."""
    corrupted = {k: v.clone() for k, v in inputs_clean.items()}
    pv = corrupted["pixel_values"]
    # Match the mean and std of the clean pixel values so the model
    # doesn't see an obviously out-of-distribution scale.
    noise = torch.randn_like(pv) * pv.std() + pv.mean()
    corrupted["pixel_values"] = noise
    return corrupted


inputs_corrupted = make_corrupted_inputs(inputs)


# ── Visualise side by side ────────────────────────────────────
def tensor_to_pil(pv_tensor):
    """Convert a pixel_values tensor [1, C, H, W] back to a displayable image."""
    t = pv_tensor[0].float().cpu()  # [C, H, W]
    t = (t - t.min()) / (t.max() - t.min())  # normalise to [0,1]
    t = (t * 255).byte().permute(1, 2, 0).numpy()
    return Image.fromarray(t)


img_clean = tensor_to_pil(inputs["pixel_values"])
img_noisy = tensor_to_pil(inputs_corrupted["pixel_values"])

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img_clean)
axes[0].axis("off")
axes[0].set_title("Clean image", fontsize=10)
axes[1].imshow(img_noisy)
axes[1].axis("off")
axes[1].set_title("Corrupted (Gaussian noise)", fontsize=10)
plt.suptitle("Activation patching setup: clean vs corrupted input", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3.3  Determine the target token id
# ============================================================
# We want to track the probability of the correct answer token ("cat")
# across the patching experiment.
#
# Note: tokenizers sometimes add a leading space before words.
# cfg["target_word"] already accounts for this (see CONFIGS above).

tokenizer = processor.tokenizer
target_word = cfg["target_word"]
target_ids = tokenizer.encode(target_word, add_special_tokens=False)

# For single-token answers, take the first id
target_id = target_ids[0]
print(f"Target word   : {repr(target_word)}")
print(f"Target token  : id={target_id}  → {repr(tokenizer.decode([target_id]))}")
print()
print("Sanity check — greedy decode of CLEAN input:")
with torch.no_grad():
    gen_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)
gen_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
print(f"  Model output: {repr(gen_text)}")

Target word   : 'cat'
Target token  : id=6635  → 'cat'

Sanity check — greedy decode of CLEAN input:
  Model output: 'USER:  \nWhat animal is in this image? Answer in one word. ASSISTANT: Cat'


## 3.4 Saving clean activations and measuring baseline probabilities

Before we can run the patching loop, we need:
1. The hidden state at every layer from the **clean run**. These are the activations we will inject.
2. The **clean probability** of the target token, which is the upper bound for restoration.
3. The **corrupted probability** of the target token, which is the lower bound (before any patching).

The restoration score at layer $L$ is then:

$$\text{restoration}(L) = \frac{P_{\text{patched at }L}(\text{cat}) - P_{\text{corrupted}}(\text{cat})}{P_{\text{clean}}(\text{cat}) - P_{\text{corrupted}}(\text{cat})}$$

A restoration of **1.0** means the patch fully recovers clean behavior.  
A restoration of **0.0** means the patch has no effect.

In practice, restoration scores can be slightly below 0 or above 1 because patching is not a perfectly linear intervention.
So values near 1 are easiest to interpret, but small overshoots are not surprising.


In [ ]:
print("target_word:", repr(target_word))
print("target_id:", target_id)
print("decoded target:", tokenizer.decode([target_id]))

for s in ["cat", " cat", "Cat", " Cat", "a", " A"]:
    ids = tokenizer.encode(s, add_special_tokens=False)
    print(repr(s), ids, [tokenizer.decode([i]) for i in ids])

target_word: 'cat'
target_id: 6635
decoded target: cat
'cat' [6635] ['cat']
' cat' [29871, 6635] ['', 'cat']
'Cat' [10459] ['Cat']
' Cat' [29871, 10459] ['', 'Cat']
'a' [263] ['a']
' A' [29871, 319] ['', 'A']


In [ ]:
import torch


def extract_hidden_from_output(output):
    """
    Extract the hidden-state tensor from a layer output.

    Supports:
      - Tensor output: returns output
      - Tuple output:  returns output[0]
    """
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, tuple):
        return output[0]
    raise TypeError(f"Unexpected layer output type: {type(output)}")


def replace_hidden_in_output(output, new_hidden):
    """
    Replace the hidden-state tensor in a layer output while preserving structure.

    Supports:
      - Tensor output: returns new_hidden
      - Tuple output:  returns (new_hidden,) + output[1:]
    """
    if isinstance(output, torch.Tensor):
        return new_hidden
    if isinstance(output, tuple):
        return (new_hidden,) + output[1:]
    raise TypeError(f"Unexpected layer output type: {type(output)}")


def validate_clean_hidden_states(clean_hidden_states, n_layers):
    """
    Ensure clean_hidden_states contains one entry for every transformer layer.
    """
    expected = set(range(n_layers))
    actual = set(clean_hidden_states.keys())
    missing = sorted(expected - actual)
    extra = sorted(actual - expected)

    if missing or extra:
        raise ValueError(
            f"clean_hidden_states validation failed.\n"
            f"Missing layers: {missing}\n"
            f"Unexpected layers: {extra}\n"
            f"Available keys: {sorted(actual)}"
        )


def get_single_token_id(tokenizer, text):
    """
    Return the token id only if text is a single token.
    Raise an error otherwise.
    """
    ids = tokenizer.encode(text, add_special_tokens=False)
    toks = tokenizer.convert_ids_to_tokens(ids)

    print(f"Target {text!r} -> ids={ids}, tokens={toks}")

    if len(ids) != 1:
        raise ValueError(f"Target {text!r} is not a single token. It splits into {len(ids)} tokens: {toks}")
    return ids[0]

In [ ]:
# ============================================================
# 3.5  Run clean and corrupted forward passes; store baselines
# ============================================================

lm_layers = get_nested_attr(model, cfg["lm_layers_path"])
n_layers = len(lm_layers)

# Optional but strongly recommended:
# ensure target_word is a SINGLE token before using target_id
# target_id = get_single_token_id(tokenizer, target_word)

# ── Clean run: save all layer hidden states ──────────────────
clean_hidden_states = {}  # layer_idx (0-based) -> [1, seq, d_model] on CPU
save_hooks = []


def make_save_hook(idx):
    def hook(module, inp, output):
        hs = extract_hidden_from_output(output)  # [B, S, D]
        clean_hidden_states[idx] = hs.detach().cpu()

    return hook


for i, layer in enumerate(lm_layers):
    h = layer.register_forward_hook(make_save_hook(i))
    save_hooks.append(h)

try:
    with torch.no_grad():
        clean_outputs = model(**inputs, return_dict=True)
finally:
    for h in save_hooks:
        h.remove()

# Check that all layers were saved
validate_clean_hidden_states(clean_hidden_states, n_layers)

print(f"Saved clean hidden states for {len(clean_hidden_states)} layers.")
print(f"Saved layer keys: {sorted(clean_hidden_states.keys())[:5]} ... {sorted(clean_hidden_states.keys())[-5:]}")

# ── Clean probability at final next-token position ───────────
clean_logits = clean_outputs.logits[0, -1].float()  # [vocab]
clean_probs = torch.softmax(clean_logits, dim=-1)
p_clean = clean_probs[target_id].item()

print(f"P({target_word!r} | clean image)     = {p_clean:.6f}")

# ── Corrupted run: baseline only, no hooks ───────────────────
with torch.no_grad():
    corrupted_outputs = model(**inputs_corrupted, return_dict=True)

corr_logits = corrupted_outputs.logits[0, -1].float()
corr_probs = torch.softmax(corr_logits, dim=-1)
p_corrupted = corr_probs[target_id].item()

print(f"P({target_word!r} | corrupted image) = {p_corrupted:.6f}")
print()
print(f"Dynamic range for restoration: [{p_corrupted:.6f}, {p_clean:.6f}]")

Saved clean hidden states for 32 layers.
Saved layer keys: [0, 1, 2, 3, 4] ... [27, 28, 29, 30, 31]
P('cat' | clean image)     = 0.000045
P('cat' | corrupted image) = 0.000008

Dynamic range for restoration: [0.000008, 0.000045]


In [ ]:
# ============================================================
# 3.6  Layer-by-layer patching loop (visual tokens only)
# ============================================================
# For each layer L (0-indexed):
#   - Register a hook that patches ONLY the visual-token positions
#     at layer L, replacing corrupted hidden states with clean ones.
#   - Run the corrupted forward pass.
#   - Compute P(target_token | patched) and the restoration score.
#
# This is a more targeted intervention than replacing the full layer output.

restoration_scores = []
patched_probs = []

# Safety checks
validate_clean_hidden_states(clean_hidden_states, n_layers)

assert visual_mask.ndim == 1, f"visual_mask should be 1D, got shape {visual_mask.shape}"
assert visual_mask.dtype == torch.bool, f"visual_mask should be bool, got dtype {visual_mask.dtype}"

print(f"Running {n_layers} patched forward passes (visual tokens only) ...")


def remove_hooks(hooks):
    for h in hooks:
        with contextlib.suppress(Exception):
            h.remove()
    hooks.clear()


# Attach stable layer indices once
for i, layer in enumerate(lm_layers):
    layer._patch_layer_idx = i


for patch_layer in tqdm(range(n_layers), desc="Patching layer"):
    patch_hooks = []

    def patch_hook(module, inp, output, target_layer=patch_layer):
        layer_idx = getattr(module, "_patch_layer_idx", None)

        if layer_idx is None:
            raise ValueError("Layer is missing _patch_layer_idx attribute.")

        # Only patch the selected layer
        if layer_idx != target_layer:
            return output

        if layer_idx not in clean_hidden_states:
            raise KeyError(
                f"Layer {layer_idx} not found in clean_hidden_states. "
                f"Available keys: {sorted(clean_hidden_states.keys())}"
            )

        current_hs = extract_hidden_from_output(output)  # [B, S, D]
        clean_hs = clean_hidden_states[layer_idx].to(device=current_hs.device, dtype=current_hs.dtype)

        if clean_hs.shape != current_hs.shape:
            raise ValueError(
                f"Shape mismatch at layer {layer_idx}: "
                f"clean_hs.shape={clean_hs.shape}, current_hs.shape={current_hs.shape}"
            )

        # visual_mask: [S]
        if clean_hs.shape[1] != visual_mask.shape[0]:
            raise ValueError(
                f"visual_mask length {visual_mask.shape[0]} does not match sequence length {clean_hs.shape[1]}"
            )

        patched_hs = current_hs.clone()
        vis_mask = visual_mask.to(current_hs.device)

        # Patch ONLY visual-token positions
        patched_hs[:, vis_mask, :] = clean_hs[:, vis_mask, :]

        return replace_hidden_in_output(output, patched_hs)

    try:
        for layer in lm_layers:
            h = layer.register_forward_hook(patch_hook)
            patch_hooks.append(h)

        with torch.no_grad():
            patched_out = model(**inputs_corrupted, return_dict=True)

    finally:
        remove_hooks(patch_hooks)

    p_logits = patched_out.logits[0, -1].float()
    p_probs = torch.softmax(p_logits, dim=-1)
    p_patch = p_probs[target_id].item()
    patched_probs.append(p_patch)

    # Normalized restoration: 0 = no help, 1 = full recovery
    denom = p_clean - p_corrupted
    restoration = 0.0 if abs(denom) < 1e-12 else (p_patch - p_corrupted) / denom
    restoration_scores.append(restoration)

print("Patching complete.")

peak_idx = int(np.argmax(restoration_scores))
print(f"Peak restoration = {restoration_scores[peak_idx]:.3f} at layer {peak_idx}")
print(f"Patched probability at peak layer = {patched_probs[peak_idx]:.8f}")

In [ ]:
# ============================================================
# 3.7  Plot the restoration curve (main result)
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

layer_idx = list(range(n_layers))  # 0-indexed

# ── Top panel: entropy (from Section 2) ──────────────────────
axes[0].plot(layer_idx, entropy_visual, color="steelblue", linewidth=2, label="Visual token entropy")
axes[0].set_ylabel("Mean entropy (nats)", fontsize=10)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_title("Logit-lens entropy (Section 2) — re-displayed for comparison", fontsize=10)

# ── Bottom panel: restoration curve ──────────────────────────
axes[1].fill_between(layer_idx, restoration_scores, alpha=0.25, color="darkorange")
axes[1].plot(
    layer_idx,
    restoration_scores,
    color="darkorange",
    linewidth=2.5,
    marker="o",
    markersize=4,
    label="Restoration score",
)

peak_layer = int(np.argmax(restoration_scores))
axes[1].axvline(peak_layer, color="red", linestyle="--", linewidth=1.5, label=f"Peak at layer {peak_layer}")
axes[1].axhline(1.0, color="gray", linestyle=":", linewidth=1)
axes[1].axhline(0.0, color="gray", linestyle=":", linewidth=1)

axes[1].set_xlabel("Transformer layer index", fontsize=10)
axes[1].set_ylabel("Restoration score", fontsize=10)
axes[1].set_title(
    f"Activation patching restoration curve — {cfg['model_name'].split('/')[-1]}\n"
    f"How much of the clean answer is recovered by patching each layer?",
    fontsize=10,
)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.05, 1.15)

plt.tight_layout()
plt.show()

print("\nSummary")
print(f"  Clean P(cat)     = {p_clean:.10f}")
print(f"  Corrupted P(cat) = {p_corrupted:.10f}")
print(f"  Peak restoration = {max(restoration_scores):.3f} at layer {peak_layer}")
print(f"  Minimum entropy  (logit lens) at layer {layer_indices[int(np.argmin(entropy_visual))]}")


Summary
  Clean P(cat)     = 0.0000452528
  Corrupted P(cat) = 0.0000075808
  Peak restoration = 3.922 at layer 14
  Minimum entropy  (logit lens) at layer 31


## 3.8 Interpreting the restoration curve

### What a "peak" means

A high restoration score at layer L means that injecting the clean hidden state at this layer is sufficient to recover much of the correct answer signal.

This suggests that the representation at this layer carries information that is highly useful for downstream answer formation.
It does not necessarily mean that all relevant information first appears only at this layer, nor that this layer is the unique bottleneck.

### Connection to the logit-lens entropy

Compare the two panels of the plot above:
- The entropy curve shows *when* visual tokens start resembling words (correlational).
- The restoration curve shows *which layer* causally determines the output (causal).

In most VLMs these curves are **broadly aligned**: the entropy minimum tends to occur in the same region as the patching peak. This is reassuring. The two independent methods are pointing at the same underlying phenomenon.

However, they need not be perfectly aligned. The entropy at a given layer reflects the local representational quality; the restoration score reflects downstream utility. A layer can have low visual entropy but not be the causal bottleneck if its information is redundantly encoded elsewhere.

### What this tells us about modality fusion

The restoration curve answers our original question:

> **Modality fusion does not happen all at once, but the most critical integration occurs in a specific window of middle-to-late transformer layers.**

The general pattern (interpret in light of your actual plot):

- **Low-restoration layers**: patching has little effect: The visual information at these positions has not yet been processed into a form that drives the final prediction.
- **Rising-restoration layers**: the model is progressively encoding the visual signal into the LM's representation space.
- **Peak layer**: the single layer where injecting clean visual activations most effectively restores correct behavior. This is the "causal bottleneck".
- **Post-peak layers**: the visual information has already been consolidated by the peak layer; replacing it later helps less because downstream layers are doing language-level reasoning on representations already shaped by earlier processing.

> **Model-specific observation**: for PaliGemma (18-layer backbone), restoration can be
> substantial even from layer 0. This makes sense: with only 18 layers total,
> there is less redundancy and each layer carries a larger share of the total processing.  
> For LLaVA-1.5 (32-layer Vicuna backbone), [Neo et al., 2024](https://arxiv.org/abs/2401.15947) (Section 4 "Causal Tracing")
> found the causal bottleneck concentrated in the lower-middle third of the network.

In [ ]:
# ============================================================
# 3.9  Optional: Which layers form the "heuristic window"?
# ============================================================
# Define the heuristic window as layers with restoration > 50% of peak.

threshold = 0.5 * max(restoration_scores)
critical = [i for i, r in enumerate(restoration_scores) if r >= threshold]

print(f"Layers with restoration ≥ 50% of peak ({threshold:.3f}):")
print(f"  Layer range : {min(critical)} – {max(critical)}")
print(f"  Count       : {len(critical)} / {n_layers} layers ({100 * len(critical) / n_layers:.0f}%)")
print()
print("This is a heuristic intervention window, not a sharply isolated bottleneck.")
print("These layers show substantial causal contribution under the chosen restoration threshold.")

Layers with restoration ≥ 50% of peak (1.961):
  Layer range : 10 – 19
  Count       : 10 / 32 layers (31%)

This is a heuristic intervention window, not a sharply isolated bottleneck.
These layers show substantial causal contribution under the chosen restoration threshold.


---
# 4) Synthesis: VLM vs LLM Interpretability

## 4.1 Side-by-side comparison

We now have enough results to compare the two tutorials directly.

| Question | LLM SAE Tutorial | VLM Tutorial (this notebook) |
|---|---|---|
| **What is the black-box problem?** | Superposition: neurons are polysemantic, features are entangled | Modality opacity: it is unclear where/when visual info becomes linguistic |
| **Primary tool** | Sparse Autoencoders (SAEs) | Logit Lens + Activation Patching |
| **What we measured** | Sparsity of feature activations, steering effects, dark matter (KL divergence) | Entropy of logit-lens distributions, restoration scores |
| **Key finding** | SAEs extract ~monosemantic features; dark matter = what SAEs miss | Visual tokens crystallize linguistically in middle-to-late layers; a specific layer window is causally critical |
| **Causal probe** | Feature steering: add $\alpha \cdot \text{direction}$ → output changes | Activation patching: swap one layer's activations → output changes |
| **Tool availability** | Pretrained SAE dictionaries available (SAELens, Gemma Scope) | No pretrained tools needed. Methods work directly on the HuggingFace model |

## 4.2 The VLM "dark matter"

In the LLM tutorial, *dark matter* referred to the fraction of activation variance not explained by the SAE.

For VLMs, there is an analogous concept, but it is richer:

### 1. The connector is largely unstudied

The MLP or linear projector that bridges vision encoder and LM is typically treated as a black box. It is trained from scratch on image-caption data and may encode non-linear visual-language alignments that neither probing nor patching fully reveals.

### 2. Visual tokens in early layers are genuinely opaque

Our logit lens showed that visual tokens in early layers have very high entropy, which the LM head cannot decode them into words. This does not mean *nothing* is there; the information may be encoded in directions orthogonal to the vocabulary manifold. SAEs applied to visual token positions (an active research frontier) may uncover this.

### 3. Cross-attention between visual and text tokens is uncharted

Standard activation patching replaces all positions at a given layer. It does not tell us which text tokens the visual tokens attend to (or vice versa). Attention knockout experiments (see [Neo et al., 2024](https://arxiv.org/abs/2401.15947)) are needed for that.

### 4. SAEs for VLMs are an open research frontier

As of early 2025, SAEs have been applied to:
- **CLIP vision encoders** in isolation: [Joseph et al., 2025 (CVPR)](https://arxiv.org/abs/2504.08729) and [Pach et al., 2025](https://arxiv.org/abs/2504.02821) (steering LLaVA via CLIP SAE)
- **Full VLM (LLaVA-NeXT) language model layers**: [Multimodal SAE, lmms-lab, 2024](https://huggingface.co/lmms-lab/llama3-llava-next-8b-hf-sae-131k)

But a comprehensive VLM SAE ecosystem — analogous to Gemma Scope for LLMs — does not yet exist.  
**This is a concrete open problem you could work on.**


## 4.3 Summary (one-slide version)

- **VLMs = Vision Encoder + Connector + Language Model**. Three separate components, each partially opaque.

- **Logit Lens**: - **Logit Lens**: apply the LM head to intermediate hidden states at visual token positions. Entropy starts high (patches are linguistically opaque at input) and drops across layers as the model extracts linguistic signal from visual tokens. The layer of minimum entropy, the "linguistic sweet spot", varies by model. For instance, in large 32-layer models it may tend to fall in the middle third; in compact models like PaliGemma (18 layers) it may appear as early
as layer 3. Three-stage pattern: alignment → crystallization → broadening.

- **Activation Patching**: corrupt the image, restore one layer at a time. The restoration curve peaks at the layer(s) where visual information is most causally concentrated. For shallow backbones, the curve may be broad (many layers contribute);
for deep backbones, a sharper peak is typical.

- **VLM dark matter**: the connector is understudied, early visual layers are opaque beyond entropy, and cross-modal attention patterns are not fully characterised. SAEs for VLMs are an active open problem.

---

### Suggested extensions (beyond the bootcamp)

1. **Repeat for different images**: Does the critical layer change for images with different complexity or domain?  
2. **Multi-layer patching** Instead of patching one layer at a time, patch a *contiguous window* and measure restoration. This reveals whether information is spread across many layers or concentrated.  
3. **Attention knockout**: block visual → text attention (or vice versa) at specific layers and measure the effect. See [Neo et al., 2024](https://arxiv.org/abs/2401.15947).  
4. **Probe the visual token representations**: Train a linear classifier on the hidden states at visual token positions across layers to predict image attributes (color, object category). Compare with the entropy curve.  
5. **Cross-model comparison**: Run both LLaVA and PaliGemma on the same image and compare their restoration curves. Do larger/smaller models show different heuristic windows?  
6. **CLIP SAE steering**: Use the pretrained CLIP SAE from [Pach et al., 2025](https://arxiv.org/abs/2504.02821) to steer a visual feature in CLIP, then measure the downstream effect on LLaVA's text output. This bridges the LLM SAE tutorial and this notebook.